In [1]:
import json
import csv
import numpy as np
import random
import datetime

# Classification task

In [2]:
csv_file = open("../../emoticon-data/classification.csv", "r")
reader = csv.DictReader(csv_file, delimiter=",")
image_data = [row for row in reader]
len(image_data)

211247

In [3]:
dims = ["anger","contempt","disgust","fear","happiness","neutral","sadness","surprise"]
cur_stat = {dim : 0.0 for dim in dims}
result = []

In [4]:
while len(result) < 20:
    sel = random.randint(1, len(image_data))
    new_stat = { dim: cur_stat[dim] + float(image_data[sel][dim]) for dim in dims}
    new_min = np.min([a for a in new_stat.values()])
    new_max = np.max([a for a in new_stat.values()])
    if new_max - new_min <= 1.0:
        cur_stat = new_stat
        result.append(image_data[sel]["file"])

In [5]:
for f in result:
    print(f)

img_align_celeba/015127.jpg
lfw/Lleyton_Hewitt/Lleyton_Hewitt_0014.jpg
img_align_celeba/095694.jpg
img_align_celeba/010419.jpg
img_align_celeba/076695.jpg
img_align_celeba/158418.jpg
img_align_celeba/086876.jpg
lfw/Gregg_Popovich/Gregg_Popovich_0005.jpg
lfw/Mirela_Manjani/Mirela_Manjani_0001.jpg
lfw/Wallace_Capel/Wallace_Capel_0001.jpg
img_align_celeba/148793.jpg
img_align_celeba/197333.jpg
img_align_celeba/189244.jpg
img_align_celeba/085226.jpg
img_align_celeba/025035.jpg
img_align_celeba/064571.jpg
lfw/Alvaro_Noboa/Alvaro_Noboa_0002.jpg
img_align_celeba/074677.jpg
img_align_celeba/012707.jpg
img_align_celeba/157491.jpg


In [68]:
id = "encode-"+datetime.datetime.now().isoformat()
print(f"Writing task definition for {id}")
with open("gen/work.json", "w") as file:
    json.dump({
        "type": "classify",
        "id": id,
        "data": [{"file": file for file in result}]
    }, file, indent=2)

Writing task definition for encode-2022-01-12T12:31:58.583033


# Recognition task

In [30]:
resultA = json.load(open('gen/resultA.json', 'r'))
resultB = json.load(open('gen/resultB.json', 'r'))

In [36]:
with open("gen/work.json", "r") as file:
    work =json.load(file)
    selected_faces = [row['file'] for row in work['data']]
len(selected_faces)

20

In [37]:
def convert_emoji_params(x):
    a1 = x[1] * 100
    a2 = x[2] * 100
    return {
            "valence": round(100 * x[0]),
            "arousal": round((a1+a2)/2),
            "potency": round(100 * x[3]),
            "contempt": round(100 * x[4]),
            "expression": round((a2-a1)/2 + 50)
        }

In [54]:
label_names = ['anger','contempt','disgust','fear','happiness','neutral','sadness','surprise']
labels = {
    face : label_names[np.argmax([row[label] for label in label_names])] 
  for face in selected_faces
  for row in image_data if row['file'] == face
}
labels

{'img_align_celeba/000838.jpg': 'contempt',
 'img_align_celeba/012499.jpg': 'surprise',
 'img_align_celeba/024857.jpg': 'contempt',
 'img_align_celeba/029376.jpg': 'happiness',
 'img_align_celeba/036568.jpg': 'disgust',
 'img_align_celeba/090689.jpg': 'disgust',
 'img_align_celeba/092099.jpg': 'neutral',
 'img_align_celeba/093560.jpg': 'surprise',
 'img_align_celeba/106786.jpg': 'anger',
 'img_align_celeba/117396.jpg': 'sadness',
 'img_align_celeba/128198.jpg': 'neutral',
 'img_align_celeba/153630.jpg': 'surprise',
 'img_align_celeba/167840.jpg': 'happiness',
 'img_align_celeba/168485.jpg': 'sadness',
 'img_align_celeba/173110.jpg': 'neutral',
 'img_align_celeba/181029.jpg': 'fear',
 'img_align_celeba/195696.jpg': 'disgust',
 'lfw/Joe_Lieberman/Joe_Lieberman_0009.jpg': 'sadness',
 'lfw/Ruano_Pascual/Ruano_Pascual_0001.jpg': 'fear',
 'lfw/Saman_Shali/Saman_Shali_0001.jpg': 'anger'}

In [58]:
data = []
for key in selected_faces:
    others = [{"file":face} for face in selected_faces if face!=key]
    data.append({
        "file": key,
        "B": convert_emoji_params(resultB[key]["x_opt"]),
        "A":  {
            "code": max(resultA[key]["x_opt"].items(), key=lambda x: x[1])[0]
        },
        "C": {
            "label": labels[key]
        },
        "decoy_sets": [
            np.random.permutation(
                np.random.choice(others, 5, replace=False).tolist() + [{"file":key}]
             ).tolist()
            for _ in range(5)
        ]
    })

In [57]:
id = "decode-"+datetime.datetime.now().isoformat()
print(f"Writing task definition for {id}")
with open("gen/work.json", "w") as file:
    json.dump({
        "type": "recognize",
        "id": id,
        "groups": ['C'],
        "data": data
    }, file, indent=2)

Writing task definition for decode-2022-01-17T10:11:47.955766
